# The Big Project begins!! — "THE PRICE IS RIGHT"

## Week 6 capstone: predict a product's price from its description

This week we build a model that estimates how much something costs, given a text description,
trained on a scrape of Amazon data.

### Order of play
- **DAY 1: Data Curation** ← *we are here*
- DAY 2: Data Pre-processing
- DAY 3: Evaluation, Baselines, Traditional ML
- DAY 4: Deep Learning and LLMs
- DAY 5: Fine-tuning a Frontier Model

### Today — Data Curation
We scrub and curate the dataset. The raw data lives on HuggingFace:
- Dataset: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023
- Per-category product metadata: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

### 💡 Business value of Data Curation

Data curation is sometimes seen as the unglamorous part of data science — but it's where the real
science happens. R&D on your **dataset** often moves performance more than the fashionable
hyper-parameter tuning we do later. So: prepare for quality time with data quality.

The curation logic lives in a small `pricer/` package next to this notebook:
- `pricer/items.py` — the `Item` data model (+ HuggingFace push/load helpers)
- `pricer/parser.py` — `parse()`: clean one raw datapoint into an `Item` (or drop it)
- `pricer/loaders.py` — `ItemLoader`: download & curate a whole category in parallel

In [ ]:
# imports

import os
import sys
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from tqdm.notebook import tqdm

# Make the local `pricer` package importable no matter where the notebook is launched from.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    possible = candidate / "lectures" / "week-six"
    if (possible / "pricer").exists():
        sys.path.insert(0, str(possible))
        break

from pricer.items import Item
from pricer.parser import parse

load_dotenv(override=True)

### Log in to HuggingFace

You need a (free) HuggingFace account and an access token. Create one at
https://huggingface.co/settings/tokens and put it in your `.env` as `HF_TOKEN=hf_...`.
If you get a *"Note"* about an environment variable being set, ignore it.

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

## Load our dataset

We start with a single category — **Appliances** — to explore.

> If you hit an error like *"trust_remote_code is no longer supported"*, run
> `!uv add --upgrade datasets==3.6.0` (or `pip install --upgrade 'datasets==3.6.0'`) in a new
> cell, restart the kernel, and try again.

In [ ]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)

In [ ]:
print(f"Number of Appliances: {len(dataset):,}")

Investigate a particular datapoint to see the raw fields we have to work with:

In [ ]:
dataset[6]

In [ ]:
# What's the most expensive item?

max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

Looks like it's going at a bargain price!! 😄

https://www.amazon.com/TurboChef-Electric-Countertop-Microwave-Convection/dp/B01D05U9NO/

### Curate with `parse`

`parse` keeps only items in the **\$0.50–\$999.49** range with enough descriptive text
(≥ 600 chars after scrubbing), strips product/model codes and noisy fields, and extracts a weight.
Items that don't qualify come back as `None` and get filtered out.

In [ ]:
# Load into Item objects if they have a price range $0.50-$999.49 and enough details
items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
items[0]

`item.full` is the cleaned text the model will learn from:

In [ ]:
print(items[0].full)

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
# Plot the distribution of text lengths
plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)

In [ ]:
# Plot the distribution of prices
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
print(items[3].full)

## Scaling up — load many categories with `ItemLoader`

`ItemLoader` downloads a whole category from HuggingFace and curates it across CPU processes.

⚠️ **Heads-up:** these steps download large datasets and pin your CPU. The full multi-category load
can take a while and a lot of memory. Pass `workers=1` to `loader.load()` if it's too heavy, or just
load a single category to follow along.

In [ ]:
from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()

In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

In [ ]:
print(f"A grand total of {len(items):,} items")

In [ ]:
items[1000]

### Deduplicate

Drop items that share a title, then items that share identical full text:

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# Plot the distribution of prices
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
category_counts = Counter([item.category for item in items])
categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.show()

### Build a balanced sample

The raw data skews toward cheap items and a couple of huge categories (Automotive,
Tools_and_Home_Improvement). We draw a **price- and category-weighted** sample so the final
dataset is less lopsided — favouring pricier items and down-weighting the dominant categories.

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# Shuffle the sample again for the final dataset
random.seed(42)
random.shuffle(sample)

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
category_counts = Counter([item.category for item in sample])
categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.show()

In [ ]:
# For another perspective, a donut chart of category share
plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')
plt.axis('equal')
plt.show()

### Any simple correlations?

Before reaching for ML, sanity-check whether price correlates with obvious features like text length or weight:

In [ ]:
# How does the price vary with the character count?
sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')
plt.show()

In [ ]:
# How does the price vary with the weight?
ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')
plt.show()

## Push this dataset to the HuggingFace Hub

Replace `username` with **your** HuggingFace username to push your own curated copy.
Or skip this cell — you can load Ed's dataset on Day 2 instead.

We save two versions: a **full** dataset (800k train) and a **lite** dataset (20k train) for faster iteration.

In [ ]:
username = "your-hf-username"  # <-- replace with your HuggingFace username
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## Sidenote

If you like matplotlib's named colors, bookmark this:
https://matplotlib.org/stable/gallery/color/named_colors.html